---
layout: article
title: The making of abandoned cities
custom_css: article.css
include_mathjax: true
publish: false
---

In [ ]:
# import necessary libs (available in jupyter/scipy-notebook docker image)
import os
import pandas as pd 
import numpy as np
import math
import matplotlib.pyplot as plt 
from matplotlib.ticker import FormatStrFormatter

# define watermark 
def add_watermark(ax, x, y):
    ax.text(ax.get_xlim()[0]+ x,
            ax.get_ylim()[0]+ y,
            "ladydragoncapital",
            alpha=0.3, fontsize=16)


# globals
HOME_DIR = '/home/jovyan/_jupyter'
DATA_DIR = os.path.join(HOME_DIR, 'data')

# read in csv data
cities = pd.read_csv(os.path.join(DATA_DIR, 'Resource_Depleted_Cities.csv'))
df = cities[:71]
df= df.round(decimals=2)
df

## Introduction ##

In some cities, an oversupply of housing—driven by the collapse of once-thriving industries—has driven real estate prices to extremely low levels. In this article, we explore what makes certain cities full of abandoned, undervalued properties. We aim to understand the origins of these cities, examine their population changes over time, and rank them based on current population and GDP.

## Historical Reasons for Undervalued Abandoned Properties ##

Most resource-depleted cities emerged as direct products of China's rapid industrialization under the planned economy. They were strategically built around abundant mineral or forest deposits—such as FX (coal), YM (oil), and GJ (tin)—to supply the nation's industrial needs. These cities developed heavily centralized, single-industry economies, making them highly dependent on finite natural resources.

Beginning in the 1980s, after decades of intensive extraction, these resources became exhausted, triggering widespread economic collapse. This natural decline was compounded by legacy issues from the planned era: state-owned enterprises that had once provided housing, healthcare, and schools went bankrupt, leading to massive layoffs and severe strain on local government finances. Severe environmental degradation further hindered any efforts at economic diversification.

In short, these cities were built for strategic national sacrifice during industrialization, only to face a painful decline as resources ran out, leaving them struggling to reinvent themselves economically.

During the boom years, these cities constructed massive housing complexes for mine workers and their families. Local governments, anticipating continued growth, approved large-scale urban expansion projects.

However, as mines closed, jobs disappeared. Many of these cities experienced both population decline and economic contraction. The working-age population—especially younger people—left en masse for larger cities in search of employment. Since the 1990s, many of these cities have lost 20–30% of their population. 

Compounding the exodus of young workers is a rapidly aging population. Those who stayed behind are predominantly elderly. The aging population not only deepens the economic malaise but also further depresses the housing market, as older residents downsize or pass away, leaving behind even more vacant properties.

This persistent oversupply of housing amid shrinking demand and an aging demographic ultimately created the conditions for deeply discounted, often abandoned real estate.

An example of this trend is shown in Figure 1: between 2000 and 2024, DXAL lost 45% of its population, and HG lost 24% of its population.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig_count = 1

# Set the style to a dark theme
plt.style.use("dark_background")

# Match website background
plt.rcParams["figure.facecolor"] = "#181818"
plt.rcParams["axes.facecolor"] = "#181818"
plt.rcParams["axes.edgecolor"] = "#181818"

# Create figure and axis
fig, ax = plt.subplots(figsize=(8, 6))

# Data
cities = ['DX', 'HX']
pop_2000 = [54.6, 109.9]
pop_2024 = [30.2, 83.4]

x = np.arange(len(cities))
width = 0.2

# Bar chart
bars1 = ax.bar(x - width/2, pop_2000, width, label='2000', color='steelblue')
bars2 = ax.bar(x + width/2, pop_2024, width, label='2024', color='coral')

# Labels and title
ax.set_xlabel('City', color='white')
ax.set_ylabel('Population (10,000 persons)', color='white')
ax.set_xticks(x)
ax.set_xticklabels(cities, color='white')
ax.tick_params(colors='white')
ax.legend(facecolor='#181818', edgecolor='white', labelcolor='white')

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}',
                xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom',
                color='white')

for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}',
                xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom',
                color='white')

# Suptitle (moved up so it's visible - y=0.0001 was pushing it off screen)
fig.suptitle(
    f"Figure {fig_count}. Population change in resource-depleted cities: 2000 vs 2024",
    fontsize=12,
    color='white',
    y=0.0001
)


# Increment counter (won't persist after script ends, but kept for logic)
fig_count += 1

plt.tight_layout()
plt.show()

## Utility Calculation ##

To systematically compare resource-depleted cities and evaluate their relative attractiveness, we construct a utility function. This function allows us to quantify each city's overall condition based on two key factors: population density and economic output (GDP per Capita).

We define the utility function as:

$$
U = \frac{1}{P +  \gamma G} 
$$

Where:

+ $P$: Population and tourist density
+ $\gamma$: coefficient for GDP per Capita
+ $G$: GDP per Capita

The utility function is designed to capture the inverse relationship between a city's vitality and its decline. A higher population density and stronger economic output suggest a more resilient city, which corresponds to a lower utility score—meaning the city is less "distressed."

Conversely, cities with shrinking populations and weak GDP per capita will have higher utility scores, indicating they are more severely affected by resource depletion and economic contraction.

As shown in Figure 1, cities ranked as the top have weak GDP and shrinking population.

In [ ]:
# Calculate population density
df['Population_Density'] = df['Population_2020'] / df['Area_km2']

# Calculate population density

# Convert both columns to numeric, coercing errors to NaN
df['GDP_2020_100million'] = pd.to_numeric(df['GDP_2020_100million'], errors='coerce')
df['Population_2020'] = pd.to_numeric(df['Population_2020'], errors='coerce')

# Now the division will work
df['GDP_per_Capita'] = df['GDP_2020_100million'] / df['Population_2020'] * 10000

# calculate cost data
#df = df.dropna()
df['Cost'] = df['Population_Density']/100 + df['GDP_per_Capita'] 
              
df['Utility'] = 1/df['Cost']*100
df

In [ ]:
# plot utility

# Sort data by Population_Density in ascending order (lowest at top)
df_sorted = df.sort_values('Utility', ascending=True)

# Create horizontal bar plot
fig, ax = plt.subplots(figsize=(8, 10))

# Create bars
bars = ax.barh(
    df_sorted['City_Initials'], 
    df_sorted['Utility'], 
    height=0.3, 
    color='goldenrod',
    edgecolor='black',
    linewidth=0.5
)

# Add value labels
ax.bar_label(bars, fmt='%.2f', padding=3, fontsize=9)

# Labels and formatting
ax.set_xlabel('Utility', fontsize=10, fontweight='bold')
ax.set_ylabel('City', fontsize=10, fontweight='bold')
ax.tick_params(axis='both', labelsize=10)



# Add grid
ax.grid(axis='x', alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

# Title
plt.suptitle(
    f"Figure {fig_count}. Utility of resource depleted cities", 
    y=-0.02, 
    fontsize=12,
    fontweight='bold'
)

plt.tight_layout()
fig_count += 1
plt.show()

## Types of Residential Buildings ##

In these cities, they built both single-family homes and multi-story apartment buildings for employees. The choice of which type to build was driven more by era and environment. 

+ Mining cities: multi-story apartment buildings

  In mining cities like HG and FX, multi-story apartment blocks dominate for the high worker density: Mines required a large workforce concentrated in specific areas. Building vertically was the most efficient way to house thousands of workers and their families in limited space. State-owned mining companies built multi-story "company dormitory" style housing. 
 
+ Forestry cities: Independent Houses

  In contrast, forestry cities in places like the DXAL region traditionally built independent houses, not large apartment complexes, because forestry work is spread across vast, remote areas. Workers are stationed at different forest farms, logging sites, and forest ranger stations. There was no single "central mine" where all workers lived, so housing was naturally built as individual, dispersed units.


Cities that had forestry industry and independent houses:
+ YC
+ BS
+ DXAL
+ AES
+ DH
+ WQ

Let's look at the utility of these forestry cities.

TODO: construct the utility funcion for forestry cities and compare them.

In [ ]:
# select data of forestry cities

# Case-sensitive
df_forestry = df[df['Original_Industry_Type'].str.contains('Forestry')]

# Case-insensitive (matches 'Forestry', 'FORESTRY', etc.)
df_forestry = df[df['Original_Industry_Type'].str.contains('Forestry', case=False)]
df_forestry

In [ ]:
# Plot population density

# Sort data by Population_Density in ascending order (lowest at top)
df_sorted = df.sort_values('Population_Density', ascending=False)

# Create horizontal bar plot
fig, ax = plt.subplots(figsize=(8, 10))

# Create bars
bars = ax.barh(
    df_sorted['City_Initials'], 
    df_sorted['Population_Density'], 
    height=0.3, 
    color='goldenrod',
    edgecolor='black',
    linewidth=0.5
)

# Add value labels
ax.bar_label(bars, fmt='%.2f', padding=3, fontsize=9)

# Labels and formatting
ax.set_xlabel('Population Density (people/km²)', fontsize=10, fontweight='bold')
ax.set_ylabel('City', fontsize=10, fontweight='bold')
ax.tick_params(axis='both', labelsize=10)



# Add grid
ax.grid(axis='x', alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

# Title
plt.suptitle(
    f"Figure {fig_count}. Population density of resource depleted cities", 
    y=-0.02, 
    fontsize=12,
    fontweight='bold'
)

plt.tight_layout()
fig_count += 1
plt.show()

In [ ]:
# plot GDP per capita
# Sort data by Population_Density in ascending order (lowest at top)
df_sorted = df.sort_values('GDP_per_Capita', ascending=False)

# Create horizontal bar plot
fig, ax = plt.subplots(figsize=(8, 10))

# Create bars
bars = ax.barh(
    df_sorted['City_Initials'], 
    df_sorted['GDP_per_Capita'], 
    height=0.3, 
    color='goldenrod',
    edgecolor='black',
    linewidth=0.5
)

# Add value labels
ax.bar_label(bars, fmt='%.2f', padding=3, fontsize=9)

# Labels and formatting
ax.set_xlabel('GDP per Capita (k yuan/person)', fontsize=10, fontweight='bold')
ax.set_ylabel('City', fontsize=10, fontweight='bold')
ax.tick_params(axis='both', labelsize=10)



# Add grid
ax.grid(axis='x', alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

# Title
plt.suptitle(
    f"Figure {fig_count}. GDP per Capita of resource depleted cities", 
    y=-0.02, 
    fontsize=12,
    fontweight='bold'
)

plt.tight_layout()
fig_count += 1
plt.show()

4000 dollars for renting and buying food, how long it can last?
vs.
4000 dollars to buy a house, set up a farm (food, chickens) and buy things I need, how long can it last